# 04s. 점수 라벨 (소프트 라벨 + 모름) 학습 파일

GLiClass 학습 파일의 `true_labels`를 **딕셔너리 `{라벨 문장: 점수}`**로 만든다 (`gliclass/data_processing.py`: 딕셔너리면 라벨마다 점수, 없는 라벨은 0, `-100`은 손실에서 빠짐).

**바꾸는 것은 학습 파일(`kshot/train_k*_s*.json`)의 라벨 값뿐이다.** 분할(pool/val/test), k-shot 예시(같은 윈도우), 검증·시험 정답(0/1)은 `data/v2/{설정}`과 똑같이 복사한다. 그래서 결과 차이는 라벨 점수의 효과만 남는다.

출력
- `data/v2/{A|B}-soft_*`: 학습 파일 = 점수 라벨
- `data/v2/{A|B}-mask_*`: 학습 파일 = 기존 0/1 그대로 (비교 기준)
- 둘 다 **검증·시험 채점은 확실한 라벨만** (7번 칸): 점수가 1인 라벨만 정답, 점수가 1이 아닌 라벨(0.7·0.5·0.2·-100) 칸은 채점에서 뺌 (에폭 고르기·임계값 고르기 포함). 그래서 두 폴더의 결과를 같은 기준으로 비교할 수 있다

예전 출력 설명: `data/v2/{A|B}-soft_{tactic|technique|both}_k{k}/` (예: `B-soft_both_k8`). `finetuning_model.ipynb`에서 `VARIANT = 'B-soft_both_k8'`로 돌린다.

## 이벤트 점수 규칙 (공격 이벤트의 기법)

| 경우 | 점수 | 근거 |
|---|---|---|
| Splunk 공격 (데이터셋 기법) | 1 | 원본 yml 기법과 97.7% 일치 (`labels/source_label_check.md`) |
| Stratus 공격 | 1 | 파일마다 기법 하나, userAgent로 공격 호출 확인 |
| Stratus 보조 호출 (`flags`에 `helper_call`: `AssumeRole`, `GetCallerIdentity`) | 0.5 | 공격 준비 호출이라 그 기법 자체인지 불확실 |
| BOTS v3 공격, TrailDiscover와 일치 (일치·상위하위 일치) | 1 | 두 출처가 일치 |
| BOTS v3 공격, TrailDiscover와 불일치 | 0.7 | 우리가 만든 규칙(주체·IP)의 라벨이고 다른 출처와 불일치 |
| TrailDiscover 검증 기법 (위에 없는 것만 추가) | 0.5 | 원본과 겹친 비율 약 0.43 |
| TrailDiscover 미검증 후보 (위에 없는 것만 추가) | 0.2 | 근거가 약함 |
| 점수가 1 미만인 기법의 **하위 기법** 중 점수가 없는 것 | -100 | 모름 (가족 범위만) |
| **상위 기법만 있는 라벨**: 시나리오 기법이 하위 기법이 있는 상위 기법이면, 그 하위 기법 중 점수가 없는 것 (점수 1이어도) | -100 | 어느 하위 기법인지 모름 (예: Splunk `T1098`, `T1552`, `T1110`) |
| 정상 이벤트 | `BENIGN` 1 | |

같은 라벨에 점수가 여럿이면 **큰 값**. 상위 기법 = 하위 기법 점수의 최댓값 (L3), 전술 = 그 전술에 속한 기법 점수의 최댓값 (L2). TrailDiscover 기법은 `01_traildiscover_labels.ipynb`와 같은 방식으로 v19 ID로 옮긴다.

## 합치는 규칙 (윈도우 5개 이벤트, 같은 내용 윈도우 병합)

라벨마다 **1 → -100 → 소프트 최댓값 → 0** 순서. 하나라도 1이면 1, 아니면 하나라도 -100이면 -100 (모르는 이벤트가 그 기법일 수 있음), 아니면 소프트 점수 중 최댓값. 공격 이벤트가 하나라도 있는 윈도우(병합 묶음)는 `BENIGN`을 뺀다 (L5, L6). B의 이벤트 반복 압축은 같은 `attack_id`·같은 API만 합치므로 첫 이벤트 점수를 그대로 쓴다.

## 학습 파일

방식(전술 / 기법 / 둘 다)에 맞는 라벨 중 그 설정의 **학습 라벨만** 남긴다 (숨김 라벨은 학습 파일에 절대 넣지 않음).

## 1. 준비

In [1]:
import json, glob, re, collections
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
L = pd.read_csv(ROOT / 'labels' / 'labels.csv', keep_default_na=False).set_index('label_id')

# TrailDiscover는 v19 이전 ID. v19에서 바뀐 ID는 이름이 같은 v19 ID로 옮긴다 (labels.csv 이름으로 확인)
CROSSWALK = {'T1562.001': 'T1685',      # Impair Defenses: Disable or Modify Tools → Disable or Modify Tools
             'T1562.008': 'T1685.002',  # Disable or Modify Cloud Logs → Disable or Modify Cloud Log
             'T1562.007': 'T1686.001',  # Disable or Modify Cloud Firewall → Cloud Firewall
             'T1562.004': 'T1686'}      # Disable or Modify System Firewall → Disable or Modify System Firewall
OLD_DEFENSE_EVASION = 'TA0005'          # v19 이전 Defense Evasion = v19의 TA0005 Stealth + TA0112 Defense Impairment

def tid(x):
    """'T1078.004 - Cloud Accounts', 'T1078.004: Valid Accounts: ...' → 'T1078.004'"""
    m = re.match(r'\s*(T[AS]?\d{4}(?:\.\d{3})?)', x)
    return m.group(1) if m else None

def parents(t):
    if t not in L.index or L.at[t, 'kind'] != 'technique': return set()
    return {t}.union(*[parents(p) for p in L.at[t, 'parent_ids'].split('|')])

def tactics_of(t):
    if t not in L.index: return set()
    if L.at[t, 'kind'] == 'tactic': return {t}
    return set().union(*[tactics_of(p) for p in L.at[t, 'parent_ids'].split('|')])

def load_td():
    rows = {}
    for f in glob.glob(str(ROOT / 'data/raw/traildiscover/repo/events/**/*.json'), recursive=True):
        d = json.load(open(f))
        raw = [tid(x) for x in d['mitreAttackTechniques'] + d['mitreAttackSubTechniques']]
        unv = [tid(x['technique']) for x in d['unverifiedMitreAttackTechniques'] if 'technique' in x]   # 2개는 형식이 다름 (description, link)
        rows[(d['eventSource'], d['eventName'])] = dict(
            td_tactics_raw=sorted({tid(x) for x in d['mitreAttackTactics']}),
            td_tech_raw=sorted(set(filter(None, raw))),
            td_tech=sorted({CROSSWALK.get(t, t) for t in raw if t}),
            td_unverified=sorted({CROSSWALK.get(t, t) for t in unv if t}),
            usedInWild=d['usedInWild'])
    return rows

import json, shutil
V2 = ROOT / 'data' / 'v2'
SCORE = dict(certain=1.0, bots_mismatch=0.7, helper=0.5, td=0.5, td_unverified=0.2, unknown=-100.0)
td = load_td()
by_name = collections.defaultdict(list)
for (s, n) in td: by_name[n].append(s)
def td_key(src, name):
    if (src, name) in td: return (src, name)
    if len(by_name.get(name, [])) == 1: return (by_name[name][0], name)
    return None
children = collections.defaultdict(set)
for t in L.index:
    if L.at[t, 'kind'] == 'technique':
        for p in L.at[t, 'parent_ids'].split('|'):
            if p in L.index and L.at[p, 'kind'] == 'technique': children[p].add(t)
ev = pd.read_csv(ROOT / 'data/interim/event_labels.csv', keep_default_na=False, low_memory=False)
ev['is_attack'] = ev.is_attack.astype(str) == 'True'
print('이벤트', len(ev), '| 공격', int(ev.is_attack.sum()), '| TrailDiscover', len(td))

이벤트 28558 | 공격 8695 | TrailDiscover 467


## 2. 이벤트 점수

In [2]:
def tech_ok(t, key):
    """우리 기법이 TrailDiscover 검증 기법과 일치하거나 상위·하위로 일치하나."""
    if key is None: return False
    V = set(td[key]['td_tech'])
    return t in V or bool(parents(t) & V) or any(t in parents(x) for x in V)

def known(t): return t in L.index and L.at[t, 'kind'] == 'technique'

def event_scores(r):
    if not r.is_attack:
        return {'BENIGN': 1.0}, 'benign'
    key = td_key(r.eventSource, r.eventName)
    if r.source == 'stratus' and 'helper_call' in r['flags']:
        base, why = SCORE['helper'], 'stratus_helper'
    elif r.source == 'botsv3':
        base, why = (SCORE['certain'], 'bots_td_agree') if tech_ok(r.technique, key) else (SCORE['bots_mismatch'], 'bots_td_mismatch')
    else:
        base, why = SCORE['certain'], f'{r.source}_dataset'
    tech = {}
    def put(t, s):
        if known(t): tech[t] = max(tech.get(t, 0.0), s)
    for t in parents(r.technique): put(t, base)                        # 시나리오 기법 + 상위 기법
    if key is not None:
        for t in td[key]['td_tech']:
            for p in parents(t) if known(t) else []: put(p, SCORE['td'])
        for t in td[key]['td_unverified']:
            for p in parents(t) if known(t) else []: put(p, SCORE['td_unverified'])
    for t, s in list(tech.items()):                                      # 상위 = 하위 최댓값
        for p in parents(t): tech[p] = max(tech.get(p, 0.0), s)
    out = dict(tech)
    if known(r.technique):                                               # 상위 기법만 있는 라벨 → 하위 기법은 모름
        for c in children[r.technique]:
            if c not in out: out[c] = SCORE['unknown']
    for t, s in tech.items():                                            # 1 미만 기법의 하위 기법 중 점수 없는 것 → 모름
        if s < 1:
            for c in children[t]:
                if c not in out: out[c] = SCORE['unknown']
    for t, s in tech.items():                                            # 전술 = 기법 점수 최댓값
        for ta in tactics_of(t): out[ta] = max(out.get(ta, 0.0), s)
    return out, why

res = ev.apply(event_scores, axis=1)
ev['scores'] = [json.dumps(x[0], sort_keys=True) for x in res]
ev['score_rule'] = [x[1] for x in res]
ev[['event_uid', 'source', 'scenario', 'eventName', 'is_attack', 'technique', 'score_rule', 'scores']].to_csv(ROOT / 'data/interim/event_scores.csv', index=False)
SC = dict(zip(ev.event_uid, [x[0] for x in res]))
print(ev[ev.is_attack].score_rule.value_counts().to_string())
vals = collections.Counter(v for x in res for v in x[0].values())
print('점수 분포 (라벨 칸 수):', dict(sorted(vals.items())))

score_rule
splunk_dataset      7747
bots_td_agree        621
stratus_dataset      296
bots_td_mismatch      16
stratus_helper        15
점수 분포 (라벨 칸 수): {-100.0: 9997, 0.2: 6526, 0.5: 8793, 0.7: 92, 1.0: 38679}


## 3. 윈도우 점수 (5개 이벤트 합치기 → 같은 내용 병합)

In [3]:
def combine(dicts):
    """라벨마다 1 → -100 → 소프트 최댓값 → 0."""
    out = {}
    for l in set().union(*dicts) if dicts else []:
        vs = [d[l] for d in dicts if l in d]
        out[l] = 1.0 if 1.0 in vs else (SCORE['unknown'] if SCORE['unknown'] in vs else max(vs))
    return out

def drop_benign_if_attack(d, has_attack):
    return {l: s for l, s in d.items() if l != 'BENIGN'} if has_attack else d

WIN = {}
for v in 'AB':
    w = pd.read_csv(ROOT / f'data/interim/windows_{v}_size5_stride1.csv', keep_default_na=False, low_memory=False)
    w['att'] = w.m1_is_attack.astype(str) == 'True'
    w['sc'] = [drop_benign_if_attack(combine([SC[u] for u in uids.split('|')]), a) for uids, a in zip(w.event_uids, w.att)]
    grp = {g: drop_benign_if_attack(combine(list(d.sc)), d.att.any()) for g, d in w.groupby('dup_group')}
    first = w[w.is_first.astype(str) == 'True']
    WIN[v] = {(s, int(i)): grp[g] for s, i, g in zip(first.session, first.window, first.dup_group)}
    rows = [{'session': s, 'window': i, 'scores': json.dumps(d, sort_keys=True)} for (s, i), d in WIN[v].items()]
    pd.DataFrame(rows).to_csv(ROOT / f'data/interim/windows_{v}_scores.csv', index=False)
    soft = sum(1 for d in WIN[v].values() if any(0 < x < 1 or x == SCORE['unknown'] for x in d.values()))
    print(f'{v}: 대표 윈도우 {len(WIN[v])} | 1이 아닌 점수가 있는 윈도우 {soft}')

A: 대표 윈도우 9287 | 1이 아닌 점수가 있는 윈도우 2614


B: 대표 윈도우 7733 | 1이 아닌 점수가 있는 윈도우 2567


## 4. 설정 폴더 만들기 (`{A|B}-soft_*`)

`labels.csv`, pool/val/test, 평가 파일은 그대로 복사하고, k-shot 학습 파일만 점수 딕셔너리로 바꾼다.

In [4]:
made = []
for base in sorted(V2.glob('[AB]_*_k*')):
    name = base.name; v, mode = name.split('_')[0], name.split('_')[1]
    soft = V2 / name.replace(f'{v}_', f'{v}-soft_', 1)
    src_sp, dst_sp = base / 'splits' / name, soft / 'splits' / soft.name
    if soft.exists():
        shutil.rmtree(soft / 'splits', ignore_errors=True)            # 결과 폴더(results/)는 지우지 않음
    (dst_sp / 'kshot').mkdir(parents=True, exist_ok=True)
    shutil.copy(base / 'labels.csv', soft / 'labels.csv')
    for f in ['pool.jsonl', 'val.jsonl', 'test.jsonl', 'val_gliclass.json', 'test_gliclass.json']:
        shutil.copy(src_sp / f, dst_sp / f)
    lab = pd.read_csv(base / 'labels.csv'); text_of = dict(zip(lab.label_id, lab.label_text))
    TRAIN = set(lab[lab.role == 'train'].label_id)
    kinds = {'tactic': {'tactic', 'benign'}, 'technique': {'technique', 'benign'}, 'both': {'tactic', 'technique', 'benign'}}[mode]
    pool = pd.read_json(src_sp / 'pool.jsonl', lines=True)
    win_of = dict(zip(pool.text, zip(pool.session, pool.window)))
    for f in sorted((src_sp / 'kshot').glob('*.json')):
        out = []
        for ex in json.load(open(f)):
            d = WIN[v][win_of[ex['text']]]
            keep = {text_of[l]: s for l, s in d.items() if l in TRAIN and (l == 'BENIGN' or L.at[l, 'kind'] in kinds)}
            out.append({'text': ex['text'], 'all_labels': ex['all_labels'], 'true_labels': keep, 'hard_labels': ex['true_labels']})
        json.dump([{k: x[k] for k in ('text', 'all_labels', 'true_labels')} for x in out], open(dst_sp / 'kshot' / f.name, 'w'), ensure_ascii=False)
        made.append((soft.name, f.name, out))
print('만든 설정 폴더', len({m[0] for m in made}), '| 학습 파일', len(made))

만든 설정 폴더 30 | 학습 파일 90


## 5. 검사

1. 학습 파일 라벨이 그 설정의 학습 라벨뿐인가 (숨김 라벨 없음)
2. 기존 정답(hard) 라벨이 모두 점수 > 0으로 들어 있나, 점수 1인 라벨은 기존 정답 안에만 있나
3. 점수 값이 규칙의 값(1, 0.7, 0.5, 0.2, -100)뿐인가
4. 검증·시험·pool 파일이 `data/v2`와 같은가, k-shot 예시(문장)가 같은가

In [5]:
import filecmp
bad = collections.Counter(); stat = collections.Counter()
for name, fn, out in made:
    lab = pd.read_csv(V2 / name / 'labels.csv'); tr_text = set(lab[lab.role == 'train'].label_text)
    for x in out:
        k = set(x['true_labels'])
        bad['숨김·후보 밖 라벨'] += not k <= tr_text
        bad['정답 라벨이 빠짐'] += not set(x['hard_labels']) <= {l for l, s in x['true_labels'].items() if s > 0}
        bad['점수 1이 정답 밖'] += not {l for l, s in x['true_labels'].items() if s == 1} <= set(x['hard_labels'])
        bad['규칙 밖 점수'] += not set(x['true_labels'].values()) <= {1.0, 0.7, 0.5, 0.2, -100.0}
        for s in x['true_labels'].values(): stat[s] += 1
    base = name.replace('-soft', ''); sp_b, sp_s = V2 / base / 'splits' / base, V2 / name / 'splits' / name
    bad['분할·평가 파일 다름'] += not all(filecmp.cmp(sp_b / f, sp_s / f, shallow=False) for f in ['pool.jsonl', 'val.jsonl', 'test.jsonl', 'val_gliclass.json', 'test_gliclass.json'])
    bad['k-shot 예시 다름'] += [e['text'] for e in json.load(open(sp_b / 'kshot' / fn))] != [e['text'] for e in out]
print({k: v for k, v in bad.items()} or '없음')
assert sum(bad.values()) == 0, '검사 실패'
print('검사 통과 | 학습 파일 라벨 칸 점수 분포:', dict(sorted(stat.items())))

{'숨김·후보 밖 라벨': 0, '정답 라벨이 빠짐': 0, '점수 1이 정답 밖': 0, '규칙 밖 점수': 0, '분할·평가 파일 다름': 0, 'k-shot 예시 다름': 0}
검사 통과 | 학습 파일 라벨 칸 점수 분포: {-100.0: 683, 0.2: 2862, 0.5: 1661, 0.7: 36, 1.0: 6682}


## 6. 기존 학습 파일과 차이 (설정별)

In [6]:
rows = []
for name, fn, out in made:
    n_soft = sum(1 for x in out for s in x['true_labels'].values() if 0 < s < 1)
    n_unk = sum(1 for x in out for s in x['true_labels'].values() if s == -100)
    n_ex = sum(1 for x in out if any(s != 1 for s in x['true_labels'].values()))
    rows.append({'설정': name, '파일': fn, '예시': len(out), '1이 아닌 라벨이 있는 예시': n_ex, '소프트 칸': n_soft, '모름(-100) 칸': n_unk})
diff = pd.DataFrame(rows)
diff[diff.파일.str.endswith('s0.json')].reset_index(drop=True)

,설정,파일,예시,1이 아닌 라벨이 있는 예시,소프트 칸,모름(-100) 칸
0,A-soft_both_k1,train_k1_s0.json,11,7,44,7
1,A-soft_both_k16,train_k16_s0.json,82,37,169,24
2,A-soft_both_k3,train_k3_s0.json,29,18,115,15
3,A-soft_both_k5,train_k5_s0.json,43,24,169,27
4,A-soft_both_k8,train_k8_s0.json,53,26,138,28
5,A-soft_tactic_k1,train_k1_s0.json,8,4,20,0
6,A-soft_tactic_k16,train_k16_s0.json,63,20,87,0
7,A-soft_tactic_k3,train_k3_s0.json,20,9,48,0
8,A-soft_tactic_k5,train_k5_s0.json,33,15,82,0
9,A-soft_tactic_k8,train_k8_s0.json,45,19,85,0


## 7. 채점 마스크 (검증·시험) + 숨김 재검사 + `-mask` 폴더

검증·시험 파일(`val.jsonl`, `test.jsonl`)에서
- `labels`: 정답 중 **점수 1인 라벨만** (원래 정답은 `labels_hard`에 남김)
- `ignore`: 점수가 1이 아닌 라벨 (0.7, 0.5, 0.2, -100). 채점·임계값·에폭 고르기에서 이 칸을 뺀다
- `is_attack`: 윈도우에 공격 이벤트가 있나 (공격 판정의 정답. 공격 라벨이 모두 `ignore`여도 공격 윈도우)

`val_gliclass.json`, `test_gliclass.json`의 `true_labels`도 `{라벨: 1, 모름: -100}`으로 바꾼다 (`train.py`의 검증 손실도 같은 기준).

**숨김 재검사**: 점수 1인 라벨만 세어 숨김 규칙(pool·val·test 중 0개, pool < k, `held_tactic`)을 다시 적용한다. 역할이 바뀌는 라벨이 있으면 `labels.csv`를 고치고 알린다.

In [7]:
def certain_split(v, d):
    rows = []
    for r in d.itertuples():
        sc = WIN[v][(r.session, int(r.window))]
        pos = [l for l in r.labels if sc.get(l) == 1.0]
        ign = sorted(l for l, s in sc.items() if s != 1.0 and l in L.index or (l == 'BENIGN' and s != 1.0))
        rows.append((pos, ign))
    out = d.copy()
    out['labels_hard'] = d['labels']
    out['labels'] = [x[0] for x in rows]
    out['ignore'] = [x[1] for x in rows]
    out['is_attack'] = d.n_attack_events > 0
    return out

role_changes, mask_stat = [], collections.Counter()
for base in sorted(V2.glob('[AB]_*_k*')):
    name = base.name; v, mode = name.split('_')[0], name.split('_')[1]; k = int(name.split('_k')[1])
    lab = pd.read_csv(base / 'labels.csv'); text_of = dict(zip(lab.label_id, lab.label_text))
    src_sp = base / 'splits' / name
    parts = {sp: certain_split(v, pd.read_json(src_sp / f'{sp}.jsonl', lines=True)) for sp in ['pool', 'val', 'test']}
    # 숨김 재검사 (점수 1 기준 개수)
    cnt = {sp: collections.Counter(l for ls in d['labels'] for l in ls) for sp, d in parts.items()}
    for r in lab.itertuples():
        if r.label_id == 'BENIGN': continue
        c = [cnt[sp].get(r.label_id, 0) for sp in ['pool', 'val', 'test']]
        new = 'held_out' if (str(r.role_reason) == 'held_tactic' or 0 in c or c[0] < k) else 'train'
        if new != r.role: role_changes.append((name, r.label_id, r.role, new, c))
    # 폴더 두 개: -soft (4번 칸에서 만든 점수 학습 파일), -mask (기존 0/1 학습 파일)
    for kind_name in ['soft', 'mask']:
        out_dir = V2 / name.replace(f'{v}_', f'{v}-{kind_name}_', 1); sp_dir = out_dir / 'splits' / out_dir.name
        if kind_name == 'mask':
            shutil.rmtree(sp_dir, ignore_errors=True); (sp_dir / 'kshot').mkdir(parents=True, exist_ok=True)
            shutil.copy(base / 'labels.csv', out_dir / 'labels.csv')
            for f in (src_sp / 'kshot').glob('*.json'): shutil.copy(f, sp_dir / 'kshot' / f.name)
        for sp, d in parts.items():
            d.to_json(sp_dir / f'{sp}.jsonl', orient='records', lines=True, force_ascii=False)
        for sp in ['val', 'test']:
            gl = json.load(open(src_sp / f'{sp}_gliclass.json'))
            for g, (pos, ign) in zip(gl, zip(parts[sp]['labels'], parts[sp]['ignore'])):
                cand = set(g['all_labels'])
                g['true_labels'] = {**{text_of[l]: -100.0 for l in ign if l in text_of and text_of[l] in cand},
                                    **{text_of[l]: 1.0 for l in pos if text_of[l] in cand}}
                mask_stat[(sp, 'ignore 칸')] += sum(1 for l in ign if l in text_of and text_of[l] in cand)
            json.dump(gl, open(sp_dir / f'{sp}_gliclass.json', 'w'), ensure_ascii=False)
print('역할이 바뀌는 라벨:', role_changes or '없음 (점수 1 기준으로 세어도 숨김 라벨 그대로)')
assert not role_changes, '역할이 바뀜: labels.csv 고치기 필요'
print('채점에서 빼는 칸 (30개 설정 × 2 폴더 합):', dict(mask_stat))

역할이 바뀌는 라벨: 없음 (점수 1 기준으로 세어도 숨김 라벨 그대로)
채점에서 빼는 칸 (30개 설정 × 2 폴더 합): {('val', 'ignore 칸'): 44608, ('test', 'ignore 칸'): 349700}


## 8. 검사 (마스크)

1. 검증·시험 정답(`labels`)은 모두 점수 1
2. 원래 정답 중 빠진 것은 모두 `ignore`에 있음
3. `-mask` 학습 파일은 `data/v2`와 같고, `-soft`·`-mask`의 검증·시험 파일은 서로 같음

In [8]:
import filecmp
bad = collections.Counter()
for base in sorted(V2.glob('[AB]_*_k*')):
    name = base.name; v = name.split('_')[0]
    sd, md_ = [V2 / name.replace(f'{v}_', f'{v}-{x}_', 1) for x in ['soft', 'mask']]
    for sp in ['val', 'test']:
        d = pd.read_json(sd / 'splits' / sd.name / f'{sp}.jsonl', lines=True)
        for r in d.itertuples():
            sc = WIN[v][(r.session, int(r.window))]
            bad['정답인데 점수 1 아님'] += any(sc.get(l) != 1.0 for l in r.labels)
            bad['빠진 정답이 ignore에 없음'] += not (set(r.labels_hard) - set(r.labels)) <= set(r.ignore)
        bad['soft·mask 검증·시험 다름'] += not filecmp.cmp(sd / 'splits' / sd.name / f'{sp}.jsonl', md_ / 'splits' / md_.name / f'{sp}.jsonl', shallow=False)
    bad['mask 학습 파일이 v2와 다름'] += not all(filecmp.cmp(f, md_ / 'splits' / md_.name / 'kshot' / f.name, shallow=False) for f in (base / 'splits' / name / 'kshot').glob('*.json'))
print({k: v for k, v in bad.items()})
assert sum(bad.values()) == 0
n_test = sum(1 for r in pd.read_json(V2 / 'B-soft_both_k8/splits/B-soft_both_k8/test.jsonl', lines=True).itertuples() if r.ignore)
print('검사 통과 | 예: B both k8 시험 윈도우 중 채점에서 빠지는 칸이 있는 윈도우', n_test)

{'정답인데 점수 1 아님': 0, '빠진 정답이 ignore에 없음': 0, 'soft·mask 검증·시험 다름': 0, 'mask 학습 파일이 v2와 다름': 0}
검사 통과 | 예: B both k8 시험 윈도우 중 채점에서 빠지는 칸이 있는 윈도우 1022
